In [ ]:
import os

import matplotlib
import numpy as np

from cil.framework import AcquisitionData, AcquisitionGeometry
from cil.io import TIFFStackReader
from cil.recon import FBP

from gvxrPython3 import gvxr
# CT reconstruction
from gvxrPython3.utils import (
    applyFiltration,
    loadSpectrum,
)

import ipywidgets as widgets

from matplotlib import pyplot as plt

# Configure matplotlib graph
font = {
    "family" : "serif",
    "size"   : 15,
}
matplotlib.rc("font", **font)

# Uncomment the line below to use LaTeX fonts
# matplotlib.rc('text', usetex=True)

### Global variables

In [ ]:
OUTPUT_PATH = "../../output_data/beam-hardening/"

if not os.path.exists(OUTPUT_PATH):
    os.makedirs(OUTPUT_PATH)

### Create an OpenGL context

In [ ]:
gvxr.createOpenGLContext()

###  Set up the detector

In [ ]:
# Detector Position and direction
gvxr.setDetectorPosition(
    20.0, 0.0, 0.0,
    "cm"
)

gvxr.setDetectorUpVector(0, 0, -1)

# Detector Pixel definitions
gvxr.setDetectorNumberOfPixels(640, 640)

gvxr.setDetectorPixelSize(
    0.5, 0.5,
    "mm"
)

In [ ]:
def setPolySpectrum(
        tube_voltage_kV: float,
        filters=None,
        tube_angle_in_deg: float=12,
        mAs = None,
        unit = "keV",
) -> dict:

    gvxr.clearFiltration()

    gvxr.setVoltage(tube_voltage_kV, "kV")
    gvxr.setTubeAngle(tube_angle_in_deg)

    if mAs:
        gvxr.setmAs(mAs)
    else:
        gvxr.setmAs(-1)

    if filters:
        applyFiltration(filters)

    energy_bins = gvxr.getEnergyBins(unit)

    photon_count = np.array(
        gvxr.getPhotonCountsPerCm2At1m(),
        dtype=np.single
    )

    #TODO: Normalise the area under
    photon_count /= (photon_count*energy_bins).sum()

    return loadSpectrum(energy_bins, photon_count, unit, False)

In [ ]:
tube_voltage_kV = 100
tube_angle_degrees = 12
max_number_of_energy_bins = 100

filter_material = "Al"
filter_thickness = 1.0
filter_unit = "mm"

# No filter
# ---------
no_filter_hist = setPolySpectrum(
    tube_voltage_kV,
    filters=None,
    tube_angle_in_deg=tube_angle_degrees,
)

no_filter_bins, no_filter_photons = zip(*no_filter_hist.items(), strict=False)

In [ ]:
plt.close()

%matplotlib inline
def custom_filter_hist(
        material: str,
        thickness: float,
        unit: str,
        *,
        tube_voltage_kV : float,
        tube_angle_in_deg : float,
):
    filter_hist = setPolySpectrum(
        tube_voltage_kV,
        filters=[[material, thickness, unit]],
        tube_angle_in_deg=tube_angle_in_deg,
    )

    return zip(*filter_hist.items(), strict=False)



@widgets.interact(
        material=["Al", "Cu", "Sn"],
        thickness=(0, 3.0, 0.5),
        unit = ["mm","cm"],
        tube_voltage_kV = widgets.fixed(tube_voltage_kV),
        tube_angle_in_deg = widgets.fixed(tube_angle_degrees),
)
def update(material = "Al", thickness = 0.5, unit = "mm"):
    filter_bins,filter_photons = custom_filter_hist(
        material,
        thickness,
        unit,
        tube_voltage_kV=tube_voltage_kV,
        tube_angle_in_deg=tube_angle_degrees
    )
    # Plot all the spectra
    fig = plt.figure(figsize= (15,5), constrained_layout=True)
    fig.supxlabel("Energy (keV)")
    fig.supylabel("Probability distribution of photons per keV")

    plt.title(f"{thickness} {unit} {material}")
    plt.step(no_filter_bins,no_filter_photons,label="No filter")
    plt.step(filter_bins,filter_photons,label=f"{thickness} {unit} {material} filter")
    plt.legend()

    plt.show()

In [ ]:
# print(gvxr.getFiltrationMaterial(), gvxr.getFiltrationThickness("mm"))
# print(gvxr.getTubeAngle(), gvxr.getPhotonCountEnergyBins())

### Create a source

In [ ]:
# Source position
gvxr.setSourcePosition(
    -20.0,  0.0, 0.0,
    "cm",
)

# gvxr.setMonoChromatic(100.0, "keV", 1000)

# Source type
gvxr.useParallelBeam()    # For a parallel source

### Create an iron cylinder as main object source

In [ ]:
gvxr.removePolygonMeshesFromSceneGraph()

cylinder_height = 20
cylinder_radius = 10

gvxr.makeCylinder(
    "Cylinder",
    24, # Number of sectors
    cylinder_height, cylinder_radius,
    "cm",
)

gvxr.addPolygonMeshAsOuterSurface("Cylinder")
gvxr.setElement("Cylinder", "Al")

gvxr.rotateNode("Cylinder", 0, 0, 90)

### Compute CT Acquisition Arguments
| Function Argument | Description |
|-------------------|-------------|
| `projection_output_path` | The path where the X-ray projections will be saved. If path is empty, data will be stored in main memory, but not saved on the disk. If path is provided, the data will be saved on the disk, and the main memory released. |
| `screenshot_output_path` | The path where the screenshots will be saved. If kept empty, not screenshot will be saved. |
| `num_of_projections` | The total number of projections to simulate. |
| `first_angle` | The rotation angle corresponding to the first projection. |
| `include_last_angle` | A boolean flag to include or exclude the last angle. It is used to calculate the angular step between successive projections. |
| `last_angle` | The number of white images used to perform the flat-field correction. If zero, then no correction will be performed. |
| `num_of_white_images_in_flat_field` | The location of the rotation centre. |
| `position_of_centre_of_rotation` | The corresponding unit of length. |
| `unit_of_length` | The rotation axis |
| `axis_of_rotation` | The upvector |
| `integrate_energy` | If true the energy fluence is returned, otherwise the number of photons is returned (default value: true) |

In [ ]:
projection_output_path = os.path.join(OUTPUT_PATH, "cupping-recons")
screenshot_output_path = ""

num_of_projections = 503

first_angle = 0
include_last_angle = False
last_angle = 180

num_of_white_images_in_flat_field = 0

position_of_centre_of_rotation = (0, 0, 0)
unit_of_length = "cm"
axis_of_rotation = (0, 0, -1)

integrate_energy = True

gvxr.computeCTAcquisition(
    projection_output_path,
    screenshot_output_path,
    num_of_projections,
    first_angle,
    include_last_angle,
    last_angle,
    num_of_white_images_in_flat_field,
    *position_of_centre_of_rotation, unit_of_length,
    *axis_of_rotation,
    integrate_energy,
)

# TODO: Show a widget for the user to select a filter with thickness to be able to compute a CT scan

In [ ]:
# Create the TIFF reader by passing the directory containing the files
reader = TIFFStackReader(
    file_name=os.path.join(OUTPUT_PATH, "cupping-recons"),
    dtype=np.float32,
)

# Read in file, and return a numpy array containing the data
data_original = reader.read()

# The data is stored as a stack of detector images, we use the CILlabels for the axes
axis_labels = [
    "angle",
    "vertical",
    "horizontal"
]

# Normalisation
# Not strictly needed as the data was already corrected
data_normalised = data_original / data_original.max()

# Prevent log of a negative value
data_normalised[data_normalised < 1e-9] = 1e-9

# Linearisation
data_absorption = -np.log(data_normalised)

In [ ]:
geometry = AcquisitionGeometry.create_Parallel3D(
    ray_direction=[1,0,0],
    detector_position=gvxr.getDetectorPosition("cm"),
    detector_direction_x=gvxr.getDetectorRightVector(),
    detector_direction_y=gvxr.getDetectorUpVector(),
    rotation_axis_position=gvxr.getCentreOfRotationPositionCT("cm"),
    rotation_axis_direction=gvxr.getRotationAxisCT(),
)

# Set the angles, remembering to specify the units
geometry.set_angles(
    np.array(gvxr.getAngleSetCT()),
    angle_unit="degree",
)

# Set the detector shape and size
geometry.set_panel(
    gvxr.getDetectorNumberOfPixels(),
    gvxr.getDetectorPixelSpacing("cm"),
)

# Set the order of the data
geometry.set_labels(axis_labels)

Working on the CPU, we cannot reconstruct the whole 3D volume for a cone beam geometry. Instead we reconstruct a single slice:

In [ ]:
# Prepare the data for the reconstruction
acquisition_data = AcquisitionData(
    data_absorption,
    deep_copy=False,
    geometry=geometry,
)

ig = acquisition_data.geometry.get_ImageGeometry()

# get slice
data_slice = acquisition_data.get_slice(vertical='centre')

```{warning}
An NVIDIA GPU is required for the reconstruction using CIL
```

In [ ]:
# Perform the FDK reconstruction
fbp =  FBP(acquisition_data, ig, backend="tigre")
recon = fbp.run()

In [ ]:
def line_profile(
        input: np.ndarray,
        first_point: tuple[float, float],
        second_point: tuple[float, float],
):
    if len(first_point) != 2 or len(second_point) != 2:
        raise ValueError("Incorrect x,y coordinates")

    input_x_size, input_y_size = input.shape
    first_x, first_y = first_point
    second_x, second_y = second_point

    if not 0 <= first_x <= input_x_size or not 0 <= first_y <= input_y_size:
        raise ValueError("first point lies outside image")

    if not 0 <= second_x <= input_x_size or not 0 <= second_y <= input_y_size:
        raise ValueError("second point lies outside image")

    number_of_points = int(np.hypot(second_x - first_x, second_y - first_y))

    line_x_values = np.linspace(
        first_x,
        second_x,
        number_of_points,
        endpoint=True,
        dtype=int,
    )

    line_y_values = np.linspace(
        first_y,
        second_y,
        number_of_points,
        endpoint=True,
        dtype=int,
    )

    return input[line_x_values, line_y_values]

In [ ]:
%matplotlib widget
plt.close()

slice_array = recon.get_slice(vertical="centre").as_array()

# Plot all the spectra
fig, ax = plt.subplots(1, 2)
fig.canvas.layout.width = '100%'

ax[0].set(title="Vertical centre slice")
ax[1].set(title="Line profile")
ax[1].grid(True)

# TODO: Add a colour bar to see the changes
# TODO: Show the image histogram
slice_image = ax[0].imshow(
    slice_array,
    cmap="grey",
    vmin=slice_array.min(),
    vmax=slice_array.max(),
)
fig.colorbar(slice_image, orientation="vertical")

clicks = []
axis_plots = []

def onclick(event):
    if event.inaxes is not ax[0]:
        return

    if len(clicks) >= 2:
        for artist in axis_plots:
            artist.remove()
        clicks.clear()
        axis_plots.clear()

    x_data, y_data = event.xdata, event.ydata
    clicks.append((x_data, y_data))

    first_point = ax[0].plot(
        x_data,
        y_data,
        "yo",
        markersize=5
    )

    axis_plots.append(first_point[0])

    if len(clicks) == 2:
        x_coordinates = [clicks[0][0], clicks[1][0]]
        y_coordinates = [clicks[0][1], clicks[1][1]]

        drawn_line = ax[0].plot(x_coordinates, y_coordinates, "y-")
        number_of_points_in_profile = int(np.hypot(x_coordinates[1] - x_coordinates[0], y_coordinates[1] - y_coordinates[0]))

        ax[1].set_xlim(
            (0,number_of_points_in_profile+1)
        )

        line_profile_plot = ax[1].plot(
            line_profile(slice_array, clicks[0], clicks[1]),
            "c-"
        )

        axis_plots.append(drawn_line[0])
        axis_plots.append(line_profile_plot[0])

    fig.canvas.draw()


cid = fig.canvas.mpl_connect('button_press_event', onclick)

#TODO: Scan stepwedge lying flat
plt.show()